# 04 — Running time backwards: reversibility and chaos

Newton's laws don't care which way time runs — so if a crystal melts, then
at some instant every velocity is flipped ($v \to -v$), the atoms must
retrace their paths exactly and **re-assemble the crystal**. This is
Loschmidt's paradox: microscopic physics is reversible, yet you have never
seen a puddle spontaneously freeze into a snowflake lattice.

The velocity Verlet integrator is *also* time-reversible, so we can
actually run this experiment — and discover the resolution of the paradox
in the process.

In [ ]:
%pip install lammps-js matplotlib

## Melt, flip, un-melt

A small 2D crystal given enough kinetic energy to melt, integrated for 500
steps with pure NVE (no thermostat — it would inject randomness and break
reversibility). Then two atom-style variables flip every velocity and we
keep integrating:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lammps import lammps

SETUP = """
units         lj
dimension     2
lattice       hex 0.9
region        box block 0 12 0 12 -0.1 0.1
create_box    1 box
create_atoms  1 box
mass          1 1.0
velocity      all create 2.6 4321 dist gaussian
pair_style    lj/cut 2.5
pair_coeff    1 1 1.0 1.0 2.5
fix           1 all nve
fix           2d all enforce2d
run           0
"""
FLIP = """
variable nvx atom -vx
variable nvy atom -vy
velocity all set v_nvx v_nvy NULL
"""

lmp = await lammps(output=None)
lmp.commands_string(SETUP)

def positions_by_id(lmp):
    # LAMMPS re-sorts atoms in memory as they move; order by atom id
    # so positions stay comparable across time.
    order = np.argsort(lmp.extract_atom("id"))
    return lmp.extract_atom("x")[order]

x0 = positions_by_id(lmp)
box = lmp.extract_box()

steps, pes = [0], [lmp.get_thermo("pe")]
def advance(n, chunk=50):
    for _ in range(n // chunk):
        lmp.command(f"run {chunk}")
        steps.append(lmp.extract_global("ntimestep"))
        pes.append(lmp.get_thermo("pe"))

advance(500, chunk=25)           # melt
lmp.commands_string(FLIP)        # time reversal
advance(500, chunk=25)           # ...and back

d = positions_by_id(lmp) - x0
L = np.array([box[1][i] - box[0][i] for i in range(3)])
d -= L * np.round(d / L)         # minimum image convention
rmsd = float(np.sqrt((d**2).sum(axis=1).mean()))
print(f"RMS distance from the initial lattice: {rmsd:.2e}")
lmp.close()

In [ ]:
plt.figure(figsize=(7, 3.4))
plt.plot(steps, pes)
plt.axvline(500, color="k", ls="--", lw=1)
plt.text(510, min(pes) + 0.05, "velocities flipped", fontsize=9)
plt.xlabel("timestep"); plt.ylabel("potential energy / atom")
plt.title("Melting, played forward and then exactly backwards")
plt.tight_layout(); plt.show()

The energy trace is a perfect mirror image and the atoms return to their
lattice sites to ~12 decimal places. Entropy decreased for 500 straight
steps — Loschmidt wins this round.

## Chaos: why you still can't unscramble an egg

Now repeat the experiment, waiting longer and longer before flipping.
Molecular dynamics is *chaotic*: the tiny floating-point roundoff of the
flip operation grows exponentially (a positive Lyapunov exponent), and
past some horizon the return trip no longer finds the crystal:

In [ ]:
results = []
for nf in [250, 1000, 2000, 4000, 8000, 16000]:
    lmp = await lammps(output=None)
    lmp.commands_string(SETUP)
    x0 = positions_by_id(lmp)
    box = lmp.extract_box()
    lmp.command(f"run {nf}")
    lmp.commands_string(FLIP)
    lmp.command(f"run {nf}")
    d = positions_by_id(lmp) - x0
    L = np.array([box[1][i] - box[0][i] for i in range(3)])
    d -= L * np.round(d / L)
    err = float(np.sqrt((d**2).sum(axis=1).mean()))
    results.append((nf, err))
    print(f"flip after {nf:>6} steps → RMS return error {err:.2e}")
    lmp.close()

In [ ]:
nf, err = zip(*results)
plt.figure(figsize=(6, 3.6))
plt.semilogy(nf, err, "o-")
plt.xlabel("steps before the velocity flip")
plt.ylabel("RMS distance from initial lattice")
plt.title("The reversibility horizon")
plt.tight_layout(); plt.show()

There is the resolution of the paradox: reversibility is exact, but it
demands *exact* information. Any error — one part in $10^{16}$ from
roundoff, or one photon's kick in a real fluid — doubles every few
collision times, and beyond that horizon the backward movie degenerates
into just another forward one. Macroscopic irreversibility is microscopic
chaos plus finite information.

Next: [05 — Thermal conductivity](05-thermal-conductivity.ipynb), a real
transport-coefficient measurement.